In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random
%matplotlib inline
%pip install graphviz

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Here we are defining each node and its value, we are also defining operations with the nodes

from typing import Any


class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self._backward = lambda: None #value used to automate backpropagation
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0.0

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) #if other is not an instance, we assume it is an int or a float (we do this in order to be able to pass integers/floats)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad # derivative of itself * grad of the output
            other.grad += 1.0 * out.grad
        out._backward = _backward # addition propagates grad

        return out

    def __neg__(self): # negate a value
        return self * -1

    def __sub__(self, other):
        return self + (-other)
        

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad # value of the other multiplying value * output grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)) # ensure the power is an int or a float
        out = Value(self.data**other, (self, ), f'**{other}')

        def _backward():
            self.grad += other * self.data**(other-1) * out.grad #d(out)/dx * dL/d(out) = dL/dx
        out._backward = _backward

        return out


    def __rmul__(self, other): #fallback if mul fails
        return self * other


    def __truediv__(self, other): #self / other
        return self*other**-1


    def tanh(self):
        x = self.data
        t = ((math.exp(2*x) - 1)/(math.exp(2*x) + 1))
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1-t**2) * out.grad # t in here represents tan(x), and the entire expression represents the derivative 1-tan(x)^2 * out.grad
        out._backward = _backward

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad #since we use the exponent in out, we define it as the exponent * the grad of the output (dout/dx = e**x) -> 1*e**x
        out._backward = _backward

        return out

    def backward(self): #implementing backprop

        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
    

a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b; e.label='e'
d = e+c; d.label='d'
f = Value(-2.0); f.label='f'
L = d * f; L.label = 'L'
#(a.__mul__(b)).__add__(c)

In [3]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{%s | data %.4f | Grad %.4f}" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [4]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)] #generate random value n weights for n neurons w = [w1, w2, ... wn]
        self.b = Value(random.uniform(-1, 1)) # bias has a random value

    def __call__(self, x):
        # w * x + b
        act = self.b
        for wi, xi in zip(self.w, x):
            act = act + wi * xi
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b] #concat the weight + bias list

class Layer:
    def __init__(self, nin, nout): #using a single neuron (nout)
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons] #call Neuron.__call__(x)
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        params = []
        for neuron in self.neurons:
            ps = neuron.parameters()
            params.extend(ps) # add the neuron parameters into the params list
        return params

class MLP: # Multi-layer perceptron

    def __init__(self, nin, nouts): # using a list of neurons to create the layer (nouts)
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self): # to save all the weights and biases that can get adjusted
        params = []
        for layer in self.layers:
            ps = layer.parameters()
            params.extend(ps) # add the neuron parameters into the params list
        return params

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0] #desired targets
n = MLP(3, [4, 4, 1]) # 3 inputs, 2 layers of 4 neurons, 1 output
ypred = [n(x) for x in xs]
ypred

[Value(data=0.6231790638204397),
 Value(data=0.6131616063799958),
 Value(data=0.5144830300776576),
 Value(data=0.6747549994618175)]

In [6]:
ypred = [n(x) for x in xs]
loss = sum(((yout-ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0))
loss

Value(data=5.143727545010014)

In [12]:
# Entire process on a single cell

for k in range(10):
    #forward pass
    ypred = [n(x) for x in xs]
    loss = sum(((yout-ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0))
    loss

    #backward pass
    for p in n.parameters():
        p.grad = 0.0 # set gradient to 0 to avoid accumulation of gradients when doing the backward pass
    loss.backward()

    #update
    for p in n.parameters():
        p.data += -0.01 * p.grad
    
    print(k, loss.data)

0 0.38606007284818805
1 0.3679225722321477
2 0.3512159000854302
3 0.33578823068688085
4 0.32150739496815867
5 0.3082579495417964
6 0.2959387281940835
7 0.28446079152307335
8 0.27374570530468284
9 0.26372409064299507
